*The second in an Exploratory Philology series on Quantified Homer.*

I thought this was going to be a straightforward post, especially after last week's counting of the Homeric hapaxes. The plan is to reproduce the frequency-binned wordlists found in Owen and Goodspeed's 1909 book *Homeric Vocabulary*. Figure 1 shows the first lists from the book, e.g. List I: Verbs Occuring 500-2,000 Times, etc. Just as in the previous post, where we counted things that appear once (hapaxes), surely we can count things that appear hundreds or even thousands of times. The same lesson applies again here: definitions matter. How we handle orthographic variation? Which words we group under the same lemma? And, again from the reproducibility angle, how can we be sure that our answers to these questions align with those of other editors, including Owen & Goodspeed but also the editors of computational lexical resources like the Ancient Greek and Latin Dependency Treebank (AGLDT).

This post aims to reproduce the first three lists from Owen & Goodspeed’s work based on available digitized and annotated Homeric texts. It also then discusses the definition issues that surface and proposes a way to rebuild this resource as a data-driven, code-based transparent and reproducible site that is in line with the original goals of the work and addresses specific criticisms of the book since its publication.

<!-- TODO: intro prose — series note ("third in Exploratory Philology's Quantified Homer series"), framing via Francese's DCC blog post [@francese_2021] and period book reviews of Owen & Goodspeed [@owen_goodspeed_1909]. Framing point: this post *derives* frequency-ranked, POS-grouped Homeric word lists from AGLDT counts, using O&G's band structure as the target shape -- it does not transcribe O&G's own lists. Scope note: Greek Lists only; see closing section on the English Lists. -->

![Fig. 1. Owen & Goodspeed, *Homeric Vocabularies* (1909), p. 3: Lists I–III.](assets/img/owen_goodspeed_p3.jpg){#fig-og-p3}

## Source note

- Scan: [Internet Archive, `homericvocabula00goodgoog`](https://archive.org/details/homericvocabula00goodgoog) (Google-digitized copy of the 1909 printing; first edition 1906).
- Local copies: `.claude/reference/owen_goodspeed_1909.pdf` (page-image PDF) and `.claude/reference/owen_goodspeed_1909_djvu.txt` (OCR text).
- **The OCR text is not reliable for the Greek word lists** and must not be used as a data source without checking against the page image. Page images can be pulled directly via the IA IIIF endpoint: `https://iiif.archive.org/iiif/homericvocabula00goodgoog%24{N}/full/full/0/default.jpg`, where `{N}` is the 0-indexed leaf (printed p. 3 = leaf 13 in this scan).
- What follows derives its own frequency-ranked, POS-grouped word lists from AGLDT counts, using O&G's band boundaries (from the Contents, p. vii) as the target shape. O&G's own headwords are used only once, at the end, as a small check against our List I–III — not reproduced as data.

## Getting Homeric word counts (AGLDT)

In [1]:
import math
import re
import unicodedata
import urllib.request

import pandas as pd
from lxml import etree
from pyuca import Collator

treebank_base = (
    'https://raw.githubusercontent.com/PerseusDL/treebank_data/master/v2.1/Greek/texts/'
)
WORKS = {"iliad": "tlg0012.tlg001", "odyssey": "tlg0012.tlg002"}


def get_words(uri):
    with urllib.request.urlopen(uri) as f:
        tree = etree.parse(f)
    return tree.getroot().xpath('.//word')


words_by_work = {
    work: get_words(f'{treebank_base}{tlg}.perseus-grc1.tb.xml')
    for work, tlg in WORKS.items()
}

print("Number of words by work:")
print({work: len(words) for work, words in words_by_work.items()})

Number of words by work:
{'iliad': 130479, 'odyssey': 105612}


## Normalizing AGLDT lemma encoding artifacts

<!-- Same normalization used in the kumpf-1984 post; see that post for discussion. -->

In [2]:
# Standalone breathing-mark prefixes (separate visible chars)
LEADING_STANDALONE = ('\u02BD', '\u02BC', '\u1FDE', '\u1FDD')

# Combining breathing-mark prefixes (zero-width modifiers); when these
# appear at the start of a lemma it is invariably an AGLDT damaged
# proper-noun encoding (combining breathing + lowercase initial),
# whose canonical form is title-cased.
LEADING_COMBINING = ('\u0313', '\u0314')

# Noisy lemmas to drop entirely
NOISE = {'???', '\u0313"\u0313'}


def normalize_lemma(lemma):
    if not lemma or lemma in NOISE:
        return None
    combining_stripped = False
    while lemma and lemma[0] in LEADING_STANDALONE + LEADING_COMBINING:
        if lemma[0] in LEADING_COMBINING:
            combining_stripped = True
        lemma = lemma[1:]
    if '~' in lemma:
        lemma = lemma.replace('~', '\u0342')  # combining circumflex
    lemma = unicodedata.normalize('NFC', lemma)
    if combining_stripped and lemma and lemma[0].islower():
        lemma = lemma[0].upper() + lemma[1:]
    return lemma or None

## Corpus-wide lemma frequency, by part of speech

<!-- This table -- lemma, corpus-wide count, coarse POS -- is the only frequency data this notebook uses. Everything below (all 18 lists) is derived from it directly; nothing is transcribed from O&G. -->

In [3]:
POS_GROUP = {
    'v': 'verb', 't': 'verb',            # finite verb, participle
    'n': 'noun',
    'a': 'adjective',
    'd': 'adverb',
    'l': 'pronoun/adjective',              # article
    'p': 'pronoun/adjective',
    'r': 'preposition',
    'c': 'conjunction/particle',
    'g': 'conjunction/particle',
    'm': 'numeral',
    'i': 'interjection',
    'e': 'interjection',
}


def parse_word(word, work):
    postag = word.attrib.get('postag', '')
    if not postag or postag.startswith('u'):  # punctuation
        return None
    lemma = normalize_lemma(word.attrib.get('lemma'))
    if not lemma:
        return None
    return {
        "work": work,
        "lemma": lemma,
        "form": word.attrib.get('form'),
        "pos_group": POS_GROUP.get(postag[0], "other"),
    }


words_df = pd.DataFrame([
    row
    for work, words in words_by_work.items()
    for word in words
    if (row := parse_word(word, work)) is not None
])

lemma_counts = words_df["lemma"].value_counts()
lemma_pos = words_df.groupby("lemma")["pos_group"].agg(lambda s: s.mode().iat[0])

agldt_freq = (
    pd.DataFrame({"count": lemma_counts, "pos_group": lemma_pos})
    .reset_index(names="lemma")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)
agldt_freq["count"] = agldt_freq["count"].astype(int)  # counts, never fractional
agldt_freq.head(20)

,lemma,count,pos_group
0,δέ,12128,conjunction/particle
1,ὁ,5834,pronoun/adjective
2,καί,5272,conjunction/particle
3,τε,4302,conjunction/particle
4,ἐγώ,2865,pronoun/adjective
5,οὐ,2689,adverb
6,εἰμί,2125,verb
7,ἐν,2073,preposition
8,ὅς,2034,pronoun/adjective
9,ὡς,1990,adverb


## Deriving Owen & Goodspeed-style lists directly from AGLDT

<!-- List number, POS bucket, frequency band, and O&G's page number below come from the Contents (p. vii) -- band *structure*, not band *contents*. Word membership in each list is computed from `agldt_freq`, not transcribed from the book. O&G's three POS buckets (Verbs; Nouns; Pronouns/adjectives/adverbs/prepositions/etc.) are coarser than AGLDT's postag-derived `pos_group`, so the third bucket absorbs everything that isn't a verb or noun. Within a band, O&G alphabetize rather than rank by frequency (cf. Fig. 1) -- pyuca's collator is used here for the same UCA-correct Greek sort as the homeric-hapaxes post. The top band in each POS group (Lists I, VII, XIII) is left open-ended above O&G's printed ceiling (2,000 / 1,000 / 10,000): those numbers describe the single most frequent word O&G happened to observe in *their* count, not a cutoff, and our own counting method should not silently drop a word for exceeding it (e.g. εἰμί, at 2,125, is obviously top-band despite technically exceeding List I's printed "2,000"). -->

In [4]:
collator = Collator()

# (list_number, pos_bucket, low, high, og_page); low <= count < high.
# high = math.inf for the top band of each POS group -- see markdown above.
OG_BAND_DEFINITIONS = [
    (1, "verb", 500, math.inf, 3),
    (2, "verb", 200, 500, 3),
    (3, "verb", 100, 200, 3),
    (4, "verb", 50, 100, 4),
    (5, "verb", 25, 50, 5),
    (6, "verb", 10, 25, 6),
    (7, "noun", 500, math.inf, 11),
    (8, "noun", 200, 500, 11),
    (9, "noun", 100, 200, 11),
    (10, "noun", 50, 100, 12),
    (11, "noun", 25, 50, 13),
    (12, "noun", 10, 25, 14),
    (13, "pronoun/adjective/adverb/preposition/etc.", 500, math.inf, 21),
    (14, "pronoun/adjective/adverb/preposition/etc.", 200, 500, 21),
    (15, "pronoun/adjective/adverb/preposition/etc.", 100, 200, 22),
    (16, "pronoun/adjective/adverb/preposition/etc.", 50, 100, 22),
    (17, "pronoun/adjective/adverb/preposition/etc.", 25, 50, 23),
    (18, "pronoun/adjective/adverb/preposition/etc.", 10, 25, 25),
]

OG_BUCKET_POS_GROUPS = {
    "verb": {"verb"},
    "noun": {"noun"},
    "pronoun/adjective/adverb/preposition/etc.": {
        "adjective", "pronoun/adjective", "adverb", "preposition",
        "conjunction/particle", "numeral", "interjection",
    },
}


def derive_list(pos_bucket, low, high):
    allowed = OG_BUCKET_POS_GROUPS[pos_bucket]
    band = agldt_freq[
        agldt_freq["pos_group"].isin(allowed)
        & (agldt_freq["count"] >= low)
        & (agldt_freq["count"] < high)
    ]
    return sorted(band["lemma"], key=collator.sort_key)


derived_lists = pd.DataFrame([
    {"list": n, "pos": pos, "band_low": low, "band_high": high, "og_page": page, "lemma": lemma}
    for n, pos, low, high, page in OG_BAND_DEFINITIONS
    for lemma in derive_list(pos, low, high)
])
derived_lists.groupby("list").size()

list
1       6
2      17
3      55
4      84
5     155
6     339
7       9
8      28
9      60
10     92
11    173
12    381
13     42
14     36
15     46
16     77
17    158
18    335
dtype: int64

### Our List I (verbs occurring 500–1,999 times)

In [5]:
derived_lists.loc[derived_lists["list"] == 1, "lemma"].tolist()

['εἰμί', 'εἶμι', 'εἶπον', 'ἔρχομαι', 'ἔχω', 'φημί']

## Checking our List I–III against O&G's printed page 3

<!-- The only O&G word data anywhere in this notebook: three sets of headwords transcribed from p. 3 and checked against the scan, used solely to sanity-check the derivation above. Not displayed as a table -- just an overlap count per list, plus the specific words that don't match (which is where the actual definitional-drift examples live, e.g. horao/eidon below). -->

In [6]:
OG_PAGE3 = {
    1: {"ἄγω", "βαίνω", "βάλλω", "δίδωμι", "εἰμί", "εἶμι", "εἶπον", "ἔρχομαι",
        "ἔχω", "ἵστημι", "ὁράω", "τίθημι", "φημί"},
    2: {"αἱρέω", "ἀμείβω", "αὐδάω", "γίγνομαι", "ἐθέλω", "ἕπω", "ἵημι",
        "ἱκνέομαι", "κεῖμαι", "κτείνω", "λείπω", "μένω", "οἶδα", "ὄλλυμι",
        "ὄρνυμι", "πείθω", "πίπτω", "φέρω", "φεύγω", "φωνέω", "χέω"},
    3: {"ἀγορεύω", "ἀΐσσω", "ἀκούω", "ἀμύνω", "ἀνώγω", "ἄρχω", "γιγνώσκω",
        "δάμνημι", "δείδω", "δέχομαι", "δύναμαι", "δύω", "ἐάω", "ἕζομαι",
        "εἴρω", "ἐλάω", "ἔοικα", "ἐρύω", "εὔχομαι", "ἧμαι", "θνήσκω",
        "ἱκάνω", "καλέω", "καλύπτω", "κελεύω", "κίω", "κλύω", "λαμβάνω",
        "λέγω", "λύω"},
}

for n in (1, 2, 3):
    ours = set(derived_lists.loc[derived_lists["list"] == n, "lemma"])
    theirs = OG_PAGE3[n]
    shared = ours & theirs
    print(f"List {n}: {len(shared)}/{len(theirs)} of O&G's headwords also fall in our derived band")
    if theirs - ours:
        print(f"  in O&G, not ours: {sorted(theirs - ours, key=collator.sort_key)}")
    if ours - theirs:
        print(f"  in ours, not O&G: {sorted(ours - theirs, key=collator.sort_key)}")

List 1: 6/13 of O&G's headwords also fall in our derived band
  in O&G, not ours: ['ἄγω', 'βαίνω', 'βάλλω', 'δίδωμι', 'ἵστημι', 'ὁράω', 'τίθημι']
List 2: 8/21 of O&G's headwords also fall in our derived band
  in O&G, not ours: ['ἀμείβω', 'αὐδάω', 'ἕπω', 'ἵημι', 'κεῖμαι', 'κτείνω', 'λείπω', 'μένω', 'ὄλλυμι', 'πίπτω', 'φεύγω', 'φωνέω', 'χέω']
  in ours, not O&G: ['ἄγω', 'βαίνω', 'βάλλω', 'δίδωμι', 'εἶδον', 'ἵστημι', 'μάχομαι', 'πρόσφημι', 'τίθημι']
List 3: 20/30 of O&G's headwords also fall in our derived band
  in O&G, not ours: ['ἀΐσσω', 'ἀμύνω', 'ἀνώγω', 'δάμνημι', 'δέχομαι', 'ἕζομαι', 'εἴρω', 'ἐλάω', 'καλύπτω', 'λέγω']
  in ours, not O&G: ['ἀμείβω', 'ἄνωγα', 'ἀπαμείβομαι', 'ἀχεύω', 'δαμάζω', 'ἐλαύνω', 'ἕπομαι', 'ἐρῶ', 'ἵημι', 'κεῖμαι', 'κτείνω', 'λείπω', 'μέμαα', 'μένω', 'μιμνήσκω', 'νοέω', 'οἴομαι', 'ὄλλυμι', 'ὁράω', 'ὀτρύνω', 'πάσχω', 'πέλω', 'πίπτω', 'προσαυδάω', 'προσεῖπον', 'ῥέζω', 'τελέω', 'τεύχω', 'τίκτω', 'φαίνω', 'φεύγω', 'φράζω', 'φωνέω', 'χαίρω', 'χέω']


### One mismatch, followed up: ὁράω

<!-- TODO prose: AGLDT lemmatizes the "see" paradigm's aorist stem separately from the present -- a real lemmatization-boundary difference from O&G, not a counting bug. Same kind of issue as the hetero/heteros case in the kumpf-1984 post. One raw <word> entry per lemma below shows the split directly: ὁρᾶτο (imperfect, Il. 1.56) tags lemma ὁράω; ἰδὼν (aorist participle, Il. 1.148) tags lemma εἶδον. -->

In [7]:
for lemma in ("ὁράω", "εἶδον"):
    row = agldt_freq.loc[agldt_freq["lemma"] == lemma]
    print(lemma, int(row["count"].iat[0]) if len(row) else 0)

for target in ("ὁράω", "εἶδον"):
    example = next(w for words in words_by_work.values() for w in words if w.attrib.get("lemma") == target)
    print(target, "->", dict(example.attrib))

ὁράω 106
εἶδον 439
ὁράω -> {'id': '8', 'form': 'ὁρᾶτο', 'lemma': 'ὁράω', 'postag': 'v3siie---', 'head': '5', 'relation': 'ADV', 'cite': 'urn:cts:greekLit:tlg0012.tlg001:1.56'}
εἶδον -> {'id': '5', 'form': 'ἰδὼν', 'lemma': 'εἶδον', 'postag': 'v-sapamn-', 'head': '6', 'relation': 'ADV', 'cite': 'urn:cts:greekLit:tlg0012.tlg001:1.148'}


### Another mismatch, followed up: ἐλάω

<!-- TODO prose: O&G print ἐλάω as their List III headword for "drive"; AGLDT lemmatizes every form of this verb (ἤλασε, ἐλαύνων, etc.) under ἐλαύνω instead -- same verb, different citation form. ἐλαύνω's AGLDT count (171) does fall in the 100-200 band, so this is a headword-naming difference, not a real frequency mismatch. -->

In [8]:
for lemma in ("ἐλάω", "ἐλαύνω"):
    row = agldt_freq.loc[agldt_freq["lemma"] == lemma]
    print(lemma, int(row["count"].iat[0]) if len(row) else 0)

ἐλάω 0
ἐλαύνω 171


## Saving the derived lists

In [9]:
agldt_freq.to_csv("data/agldt_lemma_frequency.csv", index=False)
derived_lists.to_csv("data/agldt_derived_greek_lists.csv", index=False)

## Next steps

<!-- TODO prose. Bullet scaffold only: -->

- All 18 Greek lists are now derived directly from `agldt_freq`; only Lists I–III have been checked against O&G's actual printed page (p. 3). Transcribing pp. 4–25 would extend that *check*, not the derivation itself.
- Decide how to handle lemmas whose AGLDT count undercounts a suppletive paradigm O&G treat as one headword (ὁράω/εἶδον above) — worth a systematic pass rather than one-off spot checks.
- Decide how to handle headword-naming differences where AGLDT and O&G cite the same verb under different stems (ἐλάω/ἐλαύνω above) — a lemma-alias table would resolve these without touching the derivation logic.
- ἄγω (308, verb List I) stays short of its printed 500-2,000 band even summed with all its preverb compounds (81 more across ἐξάγω, ἀνάγω, εἰσάγω, κατάγω, etc. — 389 total): unlike ὁράω/ἐλάω, this one doesn't resolve with a lemma fix and may reflect a genuinely different source count or corpus on O&G's side. ἀνώγω (1) looks like a similar defective-verb lemmatization anomaly worth checking. Both need a closer look before drawing conclusions.
- Decide how to handle the `pronoun/adjective/adverb/preposition/etc.` bucket (Lists XIII–XVIII) against AGLDT POS tags — O&G lump several `POS_GROUP` categories together; the boundary choices here (e.g. numerals, interjections) are ours, not O&G's.
- English Lists I–XVIII (pp. 33–58): mirror the Greek lists' rank/band structure but with short English glosses per headword rather than the Greek word itself. Three options to weigh:
    1. Hand-transcribe the English-list pages directly (most faithful, most manual work).
    2. Gloss our AGLDT-derived lemmas via an existing dictionary (e.g. Autenreith) and compare wording against O&G rather than assuming a match.
    3. Skip full reproduction of the English Lists and instead add short glosses only to our derived Greek Lists as a reading aid.
- Bring in Francese's DCC post [@francese_2021] and period book reviews of Owen & Goodspeed [@owen_goodspeed_1909] for the intro framing.